# 02 因子分析
对 FU 主力合约计算全部因子，通过 IC/ICIR 检验有效性，分析因子相关性与分层收益。

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

from futuresquant.data.loader import FuturesDataLoader
from futuresquant.data.universe import ContinuousContract
from futuresquant.factors.engine import FactorEngine
from futuresquant.factors.technical import (
    ROC, MOM, RSI, BollingerBand, TSMomentum,
    MACross, MACD, ADX, PriceChannel,
    ATR, NormATR, HistoricalVolatility, VolatilityRatio,
    VolumeRatio, OBV, VWAP, OpenInterestChange,
    DayOfWeek, MinuteOfDay, SessionCode, DaysToExpiry, Lagged,
)

# ── 数据源配置 ──────────────────────────────────────────────
DATA_DIR = r'I:\stock\FuturesQuant\raw_data\1min_FU'

USE_CONTINUOUS  = True        # True = 连续主力合约；False = 单合约
SINGLE_CONTRACT = 'FU2505'    # USE_CONTINUOUS=False 时生效

START_DATE = '2024-01-01'     # 起始日期
END_DATE   = '2025-04-30'     # 结束日期
# ────────────────────────────────────────────────────────────

def _ts():
    return time.strftime('%H:%M:%S')

loader = FuturesDataLoader(DATA_DIR, cache_dir=r'I:\stock\FuturesQuant\cache')

if USE_CONTINUOUS:
    cc = ContinuousContract(loader, product='FU', adjust='back', roll_n_days_before_expiry=5)

    print(f'[{_ts()}] 读取各合约 CSV / Parquet …')
    t0 = time.time()
    klines = cc.build(start=START_DATE, end=END_DATE)
    print(f'[{_ts()}] 连续合约构建完成  耗时 {time.time()-t0:.1f}s')

    print(f'[{_ts()}] 清洗换月节点附近的 volume / open_interest …')
    ROLL_BUFFER = 30
    roll_mask = klines['contract'] != klines['contract'].shift(1)
    roll_idx  = klines.index[roll_mask]
    locs = [klines.index.get_loc(ts) for ts in roll_idx]
    bad_pos = set()
    for loc in locs:
        bad_pos.update(range(max(0, loc - ROLL_BUFFER),
                             min(len(klines), loc + ROLL_BUFFER + 1)))
    bad_idx = klines.index[sorted(bad_pos)]
    klines_clean = klines.copy()
    klines_clean.loc[bad_idx, ['volume', 'open_interest']] = np.nan

    n_rolls = roll_mask.sum()
    print(f'[{_ts()}] 完成')
    print(f'\n  模式        : 连续主力合约（后复权）')
    print(f'  时间范围    : {START_DATE} ~ {END_DATE}')
    print(f'  K线行数     : {len(klines):,}')
    print(f'  换月次数    : {n_rolls}')
    print(f'  清洗 bar 数 : {len(bad_idx):,}')
    print(f'  涉及合约    : {klines["contract"].unique().tolist()}')
else:
    print(f'[{_ts()}] 读取单合约 {SINGLE_CONTRACT} …')
    t0 = time.time()
    klines = loader.load(SINGLE_CONTRACT, start=START_DATE, end=END_DATE)
    klines_clean = klines.copy()
    print(f'[{_ts()}] 完成  耗时 {time.time()-t0:.1f}s')
    print(f'\n  模式     : 单合约')
    print(f'  合约     : {SINGLE_CONTRACT}')
    print(f'  时间范围 : {START_DATE} ~ {END_DATE}')
    print(f'  K线行数  : {len(klines):,}')

## 1. 批量计算因子

In [ ]:
FACTORS = [
    # ── 原有技术因子 ──────────────────────────────────────
    ROC(20), MOM(20), RSI(14), BollingerBand(20), TSMomentum(5, 20),
    MACross(5, 20), MACD(12, 26, 9), ADX(14), PriceChannel(20),
    ATR(14), NormATR(14), HistoricalVolatility(240), VolatilityRatio(20, 120),
    VolumeRatio(20), OBV(20), VWAP(60, 14), OpenInterestChange(20),

    # ── 时间特征 ─────────────────────────────────────────
    DayOfWeek(),       # 星期几 0-4
    MinuteOfDay(),     # 日内位置 0-1（归一化分钟）
    SessionCode(),     # 交易时段 0=夜盘 1=早盘 2=午前 3=午后
    DaysToExpiry(),    # 距交割月天数（需连续合约 contract 列）

    # ── 滞后特征（对高 IC 因子延伸时序信息）────────────
    Lagged(ROC(20),              lag=1),
    Lagged(ROC(20),              lag=5),
    Lagged(VolatilityRatio(20, 120), lag=1),
    Lagged(VolatilityRatio(20, 120), lag=5),
]

engine = FactorEngine(FACTORS)
# klines_clean：换月节点附近的 volume/open_interest 已置 NaN，
# 使 OBV / OIChange 在换月处自动产生 NaN 而非跳变值；
# 价格列与原始 klines 相同，不影响其余因子。
factor_df = engine.compute(klines_clean)

print(f'因子矩阵: {factor_df.shape}')
factor_df.describe().round(4)

## 2. IC 分析（前视 N 根 bar 的收益相关性）

**Rank IC** = Spearman 相关系数(因子_t, 收益_{t → t+N})
- |IC| > 0.02 视为有效
- **ICIR** = IC均值 / IC标准差，衡量稳定性，>0.5 为佳

In [3]:
def calc_ic(factor_df: pd.DataFrame, klines: pd.DataFrame,
            forward_bars: int = 60) -> pd.DataFrame:
    """计算每个因子的全样本 Rank IC，返回 (factor × IC/p_value) DataFrame。"""
    fwd_ret = klines['close'].pct_change(forward_bars).shift(-forward_bars)
    daily_factor = factor_df.resample('1D').last()
    daily_ret = fwd_ret.resample('1D').last()

    ic_records = {}
    for col in daily_factor.columns:
        aligned = pd.concat([daily_factor[col], daily_ret], axis=1).dropna()
        if len(aligned) < 5:
            continue
        rho, pval = stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
        ic_records[col] = {'IC': rho, 'p_value': pval}

    return pd.DataFrame(ic_records).T


def calc_rolling_ic(factor_series: pd.Series, klines: pd.DataFrame,
                    forward_bars: int = 60, window_days: int = 20) -> pd.Series:
    """计算单因子的滚动 Rank IC 序列（按日）。"""
    fwd_ret = klines['close'].pct_change(forward_bars).shift(-forward_bars)
    daily_f = factor_series.resample('1D').last().dropna()
    daily_r = fwd_ret.resample('1D').last()
    combined = pd.concat([daily_f, daily_r], axis=1).dropna()
    combined.columns = ['factor', 'ret']

    rolling_ic = []
    for i in range(window_days, len(combined)):
        window = combined.iloc[i - window_days:i]
        rho, _ = stats.spearmanr(window['factor'], window['ret'])
        rolling_ic.append((combined.index[i], rho))

    if not rolling_ic:
        return pd.Series(dtype=float)
    times, vals = zip(*rolling_ic)
    return pd.Series(vals, index=pd.DatetimeIndex(times), name='rolling_IC')


ic_summary = calc_ic(factor_df, klines, forward_bars=60)
ic_summary['|IC|'] = ic_summary['IC'].abs()
ic_summary = ic_summary.sort_values('|IC|', ascending=False)
ic_summary.round(4)

,IC,p_value,|IC|
OBV_20,-0.0344,0.2656,0.0344
ROC_20,0.0313,0.3113,0.0313
OIChange_20,-0.0280,0.3661,0.0280
HV_240,0.0245,0.4292,0.0245
MOM_20,0.0230,0.4564,0.0230
MACross_ema_5_20,0.0218,0.4808,0.0218
NATR_14,0.0213,0.4918,0.0213
BB_pct_20_2.0,0.0211,0.4947,0.0211
Channel_20,0.0202,0.5130,0.0202
RSI_14,0.0177,0.5683,0.0177


In [4]:
# IC 条形图
colors = ['#d62728' if x < 0 else '#1f77b4' for x in ic_summary['IC']]
fig = go.Figure(go.Bar(
    x=ic_summary.index, y=ic_summary['IC'],
    marker_color=colors,
    error_y=dict(type='data', array=[0]*len(ic_summary), visible=False),
))
fig.add_hline(y=0.02, line_dash='dash', line_color='green', annotation_text='IC=0.02')
fig.add_hline(y=-0.02, line_dash='dash', line_color='green')
fig.update_layout(title='各因子 Rank IC（forward 60bar）',
                   xaxis_title='因子', yaxis_title='IC', height=400)
fig.show()

## 3. 最佳因子的滚动 IC

In [5]:
top_factor_name = ic_summary.index[0]
print(f'IC 最高因子: {top_factor_name}  IC={ic_summary.loc[top_factor_name, "IC"]:.4f}')

rolling_ic = calc_rolling_ic(factor_df[top_factor_name], klines, forward_bars=60)

fig = go.Figure()
fig.add_trace(go.Scatter(x=rolling_ic.index, y=rolling_ic.values,
                          mode='lines', name='20日滚动 IC',
                          line=dict(color='steelblue')))
fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.add_hline(y=rolling_ic.mean(), line_dash='dash', line_color='orange',
               annotation_text=f'均值={rolling_ic.mean():.3f}')
fig.update_layout(title=f'{top_factor_name} 滚动 Rank IC', height=350)
fig.show()

icir = rolling_ic.mean() / rolling_ic.std()
print(f'ICIR: {icir:.4f}   IC>0 比例: {(rolling_ic > 0).mean():.1%}')

IC 最高因子: OBV_20  IC=-0.0344


ICIR: -0.1819   IC>0 比例: 40.9%


## 4. 因子相关性热图

In [6]:
corr = factor_df.dropna().corr(method='spearman').round(2)

fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, aspect='auto',
                title='因子 Spearman 相关性矩阵')
fig.update_layout(height=600, width=700)
fig.show()

## 5. 因子分层回测
将每根 bar 的因子值按五分位分组，计算各组下 60 bar 的平均收益。
理想情况下，第 5 组 > 第 1 组（单调性）。

In [7]:
def quantile_return(factor_series: pd.Series, klines: pd.DataFrame,
                    forward_bars: int = 60, n_quantiles: int = 5) -> pd.DataFrame:
    """因子分层：计算各分位组的平均前瞻收益。"""
    fwd_ret = klines['close'].pct_change(forward_bars).shift(-forward_bars)
    df = pd.concat([factor_series, fwd_ret], axis=1).dropna()
    df.columns = ['factor', 'fwd_ret']

    # 按日截面分位（避免时序污染）
    daily_last = df.resample('1D').last().dropna()
    daily_last['quantile'] = pd.qcut(daily_last['factor'], n_quantiles,
                                      labels=False, duplicates='drop') + 1
    return daily_last.groupby('quantile')['fwd_ret'].agg(['mean', 'std', 'count'])


# 对 IC 最高的几个因子做分层
top3 = ic_summary.index[:3].tolist()
fig = make_subplots(rows=1, cols=len(top3),
                    subplot_titles=[f'{f}\nIC={ic_summary.loc[f,"IC"]:.3f}' for f in top3])

for col_i, fname in enumerate(top3):
    qret = quantile_return(factor_df[fname], klines)
    fig.add_trace(
        go.Bar(x=[f'Q{int(q)}' for q in qret.index],
               y=qret['mean'] * 100,
               error_y=dict(type='data', array=qret['std'] * 100),
               name=fname, showlegend=False),
        row=1, col=col_i + 1
    )

fig.update_yaxes(title_text='平均收益 (%)', row=1, col=1)
fig.update_layout(title='因子分层平均收益（5分位，forward 60bar）', height=380)
fig.show()

## 6. 因子自相关性（衰减速度）

In [9]:
lags = range(1, 61)  # 1 ~ 60 分钟 lag

fig = go.Figure()
for fname in top3:
    f = factor_df[fname].dropna()
    autocorr = [f.autocorr(lag) for lag in lags]
    fig.add_trace(go.Scatter(x=list(lags), y=autocorr, name=fname, mode='lines'))

fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.update_layout(
    title='因子自相关性衰减（衰减越慢 → 因子持续时间越长）',
    xaxis_title='Lag (分钟)', yaxis_title='自相关系数', height=380
)
fig.show()

## 7. IC 多周期衰减（Factor Decay）
计算不同前瞻窗口（1~120 bar）的 Rank IC，衰减越慢说明因子信号持续时间越长，对应更长的持仓周期。

In [ ]:
DECAY_WINDOWS = [1, 5, 10, 20, 30, 60, 90, 120]

def calc_ic_at_window(factor_df: pd.DataFrame, klines: pd.DataFrame,
                      forward_bars: int) -> pd.Series:
    fwd_ret = klines['close'].pct_change(forward_bars).shift(-forward_bars)
    daily_f = factor_df.resample('1D').last()
    daily_r = fwd_ret.resample('1D').last()
    records = {}
    for col in daily_f.columns:
        aligned = pd.concat([daily_f[col], daily_r], axis=1).dropna()
        if len(aligned) < 10:
            records[col] = np.nan
            continue
        rho, _ = stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
        records[col] = rho
    return pd.Series(records)

decay_df = pd.DataFrame(
    {w: calc_ic_at_window(factor_df, klines, w) for w in DECAY_WINDOWS}
).T
decay_df.index.name = 'forward_bars'

top5 = ic_summary.index[:5].tolist()

fig = go.Figure()
for fname in top5:
    fig.add_trace(go.Scatter(
        x=DECAY_WINDOWS, y=decay_df[fname].values,
        mode='lines+markers', name=fname
    ))
fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.add_hline(y=0.02, line_dash='dash', line_color='green', annotation_text='IC=0.02')
fig.add_hline(y=-0.02, line_dash='dash', line_color='green')
fig.update_layout(
    title='IC 多周期衰减曲线（Top5 因子）',
    xaxis_title='前瞻窗口（bar）', yaxis_title='Rank IC', height=420
)
fig.show()

print("\n各窗口 IC 汇总（Top5 因子）:")
print(decay_df[top5].round(4).to_string())

## 8. 多空净值曲线（Long-Short Portfolio）
滚动 60 日窗口确定分位（避免前视偏差）：因子值处于高分位做多、低分位做空，跟踪净值并计算年化收益、夏普比、胜率。

In [ ]:
def long_short_nav(factor_series: pd.Series, klines: pd.DataFrame,
                   forward_days: int = 1, n_quantiles: int = 5,
                   roll_window: int = 60) -> pd.DataFrame:
    """
    滚动分位多空策略净值（日频）。
    - 使用日频收盘价计算前瞻收益，避免用 1-min gap 收益代替日收益
    - 滚动 roll_window 天确定当前因子的百分位排名
    - top 1/n_quantiles 做多，bottom 1/n_quantiles 做空
    """
    daily_close = klines['close'].resample('1D').last()
    fwd_ret = daily_close.pct_change(forward_days).shift(-forward_days)
    daily_f = factor_series.resample('1D').last()
    df = pd.concat([daily_f, fwd_ret], axis=1).dropna()
    df.columns = ['factor', 'fwd_ret']

    threshold = 1.0 / n_quantiles
    df['pct_rank'] = df['factor'].rolling(roll_window).rank(pct=True)
    df['signal'] = 0
    df.loc[df['pct_rank'] >= (1 - threshold), 'signal'] = 1
    df.loc[df['pct_rank'] <= threshold, 'signal'] = -1

    df['ls_ret'] = df['signal'].shift(1) * df['fwd_ret']
    df['nav'] = (1 + df['ls_ret'].fillna(0)).cumprod()
    df['bh_nav'] = (1 + df['fwd_ret'].fillna(0)).cumprod()
    return df


top3 = ic_summary.index[:3].tolist()
fig = make_subplots(rows=1, cols=len(top3),
                    subplot_titles=top3, shared_yaxes=False)

print(f"{'因子':<25} {'年化收益':>10} {'夏普比':>8} {'胜率':>8} {'最大回撤':>10}")
print('-' * 65)

for col_i, fname in enumerate(top3):
    nav_df = long_short_nav(factor_df[fname], klines, forward_days=1)

    ls = nav_df['ls_ret'].dropna()
    annual_ret = ls.mean() * 252
    annual_std = ls.std() * np.sqrt(252)
    sharpe = annual_ret / annual_std if annual_std > 0 else 0
    win_rate = (ls > 0).mean()
    nav_vals = nav_df['nav'].dropna()
    max_dd = ((nav_vals.cummax() - nav_vals) / nav_vals.cummax()).max()

    print(f'{fname:<25} {annual_ret:>10.2%} {sharpe:>8.3f} {win_rate:>8.1%} {max_dd:>10.2%}')

    fig.add_trace(go.Scatter(x=nav_df.index, y=nav_df['nav'],
                             name=f'{fname} L/S', line=dict(width=1.8)),
                  row=1, col=col_i + 1)
    fig.add_trace(go.Scatter(x=nav_df.index, y=nav_df['bh_nav'],
                             name='Buy&Hold', line=dict(dash='dot', color='gray', width=1),
                             showlegend=(col_i == 0)),
                  row=1, col=col_i + 1)

fig.update_layout(title='多空净值曲线（Top3 因子 | 5分位 L/S | forward 1D | 滚动60日分位）',
                  height=400)
fig.show()

## 9. 因子换手率（Turnover）
换手率 = 1 − 相邻截面因子排名的 Spearman 相关系数（取值 0~2，越高信号变化越剧烈）。  
IC/换手率 越高 → 每单位交易成本能获取的 IC 越多 → 因子越实用。

In [ ]:
def calc_turnover(factor_df: pd.DataFrame) -> pd.Series:
    """各因子日度换手率：1 - rank_autocorr(lag=1)。"""
    daily_f = factor_df.resample('1D').last()
    result = {}
    for col in daily_f.columns:
        f = daily_f[col].dropna()
        rank = f.rank(pct=True)
        result[col] = 1 - rank.autocorr(lag=1)
    return pd.Series(result).sort_values()


turnover = calc_turnover(factor_df)

summary = ic_summary[['IC', '|IC|']].copy()
summary['Turnover'] = turnover
summary['IC/Turnover'] = (summary['|IC|'] / summary['Turnover']).round(4)
summary = summary.sort_values('IC/Turnover', ascending=False)

print("因子综合评分（按 IC/换手率 排序）")
print(summary.round(4).to_string())

# 散点图：|IC| vs 换手率，气泡大小 = IC/Turnover
fig = px.scatter(
    summary.reset_index().rename(columns={'index': 'factor'}),
    x='Turnover', y='|IC|',
    text='factor',
    size=summary['IC/Turnover'].clip(lower=0.001).values,
    size_max=30,
    color='IC/Turnover',
    color_continuous_scale='RdYlGn',
    title='因子 |IC| vs 换手率（右上角为优质因子，气泡大小=IC/换手率）',
    labels={'Turnover': '换手率', '|IC|': '|IC|'},
)
fig.update_traces(textposition='top center')
fig.update_layout(height=480)
fig.show()

## 10. 交易成本盈亏平衡分析（Break-even Cost）
盈亏平衡手续费率 = `|IC| × σ_日收益 / (2 × 换手率)`  
高于此值的手续费将使因子净收益变负；结合典型 FU 手续费（~0.01~0.02% 单边），判断因子是否可执行。

In [ ]:
# FU 典型手续费参数（可按实际调整）
FU_COMMISSION_RATE = 0.0001   # 单边 1 bp（约 3~5 元/手，价格~4000，乘数 10）
FU_SLIPPAGE_RATE  = 0.0001   # 单边 1 bp 滑点

# 用日频收盘价计算日波动率，而非 1-min gap 收益
sigma_daily = klines['close'].resample('1D').last().pct_change().std()

be_df = pd.DataFrame({'|IC|': ic_summary['|IC|'], 'Turnover': turnover})
be_df['BreakEven_bps'] = (
    be_df['|IC|'] * sigma_daily / (2 * be_df['Turnover']) * 1e4
).round(2)

total_cost = FU_COMMISSION_RATE + FU_SLIPPAGE_RATE
be_df['NetIC'] = (
    be_df['|IC|'] - 2 * be_df['Turnover'] * total_cost / sigma_daily
).round(4)
be_df['Executable'] = be_df['NetIC'] > 0

print(f'日收益标准差: {sigma_daily:.4%}   单边假设成本: {total_cost:.4%}')
print()
print(be_df.sort_values('BreakEven_bps', ascending=False).round(4).to_string())

fig = go.Figure()
fig.add_bar(
    x=be_df.sort_values('BreakEven_bps', ascending=False).index,
    y=be_df.sort_values('BreakEven_bps', ascending=False)['BreakEven_bps'],
    marker_color=[('#2ca02c' if v else '#d62728') for v in
                  be_df.sort_values('BreakEven_bps', ascending=False)['Executable']],
    name='盈亏平衡手续费上限 (bps)'
)
fig.add_hline(y=(FU_COMMISSION_RATE + FU_SLIPPAGE_RATE) * 1e4,
              line_dash='dash', line_color='black',
              annotation_text=f'假设单边成本 {(FU_COMMISSION_RATE+FU_SLIPPAGE_RATE)*1e4:.1f} bps')
fig.update_layout(
    title='各因子盈亏平衡手续费上限（绿色=可执行，红色=成本吃掉收益）',
    xaxis_title='因子', yaxis_title='盈亏平衡上限 (bps)', height=420
)
fig.show()

## 11. 市场状态条件 IC（Regime-Conditional IC）
按日收益方向 + ADX 强度将市场划分为四种状态：**趋势上涨 / 趋势下跌 / 高波动震荡 / 低波动震荡**，
分别计算各因子 IC，找出每种状态下最有效的因子。

In [ ]:
def classify_regime(factor_df: pd.DataFrame, klines: pd.DataFrame,
                    trend_threshold: float = 25.0) -> pd.Series:
    """
    市场状态分类（日频）：趋势上涨 / 趋势下跌 / 高波震荡 / 低波震荡
    使用 ADX（趋势强度）+ 20日滚动收益方向。
    """
    adx_daily = factor_df['ADX_14'].resample('1D').last()
    ret_20d = klines['close'].resample('1D').last().pct_change(20)

    regime = pd.Series('低波震荡', index=adx_daily.index)
    trending = adx_daily >= trend_threshold
    mask_trend_up   = trending & (ret_20d >= 0)
    mask_trend_down = trending & (ret_20d < 0)
    mask_vol_high   = (~trending) & (adx_daily >= adx_daily.median())
    regime[mask_trend_up]   = '趋势上涨'
    regime[mask_trend_down] = '趋势下跌'
    regime[mask_vol_high]   = '高波震荡'
    return regime.dropna()


def regime_ic(factor_df: pd.DataFrame, klines: pd.DataFrame,
              regime: pd.Series, forward_bars: int = 60) -> pd.DataFrame:
    fwd_ret = klines['close'].pct_change(forward_bars).shift(-forward_bars)
    daily_f = factor_df.resample('1D').last()
    daily_r = fwd_ret.resample('1D').last()

    records = []
    for state in regime.unique():
        idx = regime[regime == state].index
        f_sub = daily_f.loc[daily_f.index.isin(idx)]
        r_sub = daily_r.loc[daily_r.index.isin(idx)]
        for col in daily_f.columns:
            aligned = pd.concat([f_sub[col], r_sub], axis=1).dropna()
            if len(aligned) < 5:
                continue
            rho, _ = stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
            records.append({'regime': state, 'factor': col, 'IC': rho})

    df = pd.DataFrame(records)
    return df.pivot(index='factor', columns='regime', values='IC')


regime = classify_regime(factor_df, klines)
print("市场状态分布:")
print(regime.value_counts().to_string())
print()

regime_ic_df = regime_ic(factor_df, klines, regime, forward_bars=60)

fig = px.imshow(
    regime_ic_df.round(3),
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-0.3, zmax=0.3,
    aspect='auto',
    title='条件 IC 热图：不同市场状态下各因子表现（forward 60 bar）'
)
fig.update_layout(height=550, width=700,
                  xaxis_title='市场状态', yaxis_title='因子')
fig.show()

print("\n条件 IC 汇总:")
print(regime_ic_df.round(4).to_string())

## 12. 分时效应（Intraday Seasonality）
FU 燃油期货交易时段：夜盘 21:00–23:00、早盘 09:00–10:15、午前 10:30–11:30、午后 13:30–15:00。  
分别计算各时段的 Rank IC（forward 1 bar），找出因子在哪个时段最有效。

In [ ]:
def assign_session(idx: pd.DatetimeIndex) -> pd.Series:
    """将每根 bar 的时间归入交易时段。"""
    t = idx.time
    import datetime
    s = pd.Series('其他', index=idx)
    s[pd.Series([(datetime.time(21, 0) <= x or x < datetime.time(2, 30)) for x in t], index=idx)] = '夜盘'
    s[pd.Series([(datetime.time(9, 0) <= x <= datetime.time(10, 15)) for x in t], index=idx)] = '早盘'
    s[pd.Series([(datetime.time(10, 30) <= x <= datetime.time(11, 30)) for x in t], index=idx)] = '午前'
    s[pd.Series([(datetime.time(13, 30) <= x <= datetime.time(15, 0)) for x in t], index=idx)] = '午后'
    return s


SESSIONS = ['夜盘', '早盘', '午前', '午后']
FORWARD_1BAR = 1

session_labels = assign_session(factor_df.index)
fwd_ret_1bar = klines['close'].pct_change(FORWARD_1BAR).shift(-FORWARD_1BAR)

session_ic_records = []
for sess in SESSIONS:
    mask = session_labels == sess
    f_sub = factor_df.loc[mask]
    r_sub = fwd_ret_1bar.loc[mask]
    for col in factor_df.columns:
        aligned = pd.concat([f_sub[col], r_sub], axis=1).dropna()
        if len(aligned) < 20:
            continue
        rho, _ = stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
        session_ic_records.append({'session': sess, 'factor': col, 'IC': rho,
                                    'n': len(aligned)})

session_ic_df = (pd.DataFrame(session_ic_records)
                  .pivot(index='factor', columns='session', values='IC')
                  .reindex(columns=SESSIONS))

# 热图
fig = px.imshow(
    session_ic_df.round(3),
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-0.05, zmax=0.05,
    aspect='auto',
    title='分时 IC 热图（forward 1 bar | 各交易时段）'
)
fig.update_layout(height=550, width=600,
                  xaxis_title='时段', yaxis_title='因子')
fig.show()

# 各时段样本量
n_df = (pd.DataFrame(session_ic_records)
         .groupby('session')['n'].mean()
         .reindex(SESSIONS))
print("\n各时段平均样本 bar 数:")
print(n_df.round(0).to_string())

## 13. 换月/交割效应（Roll Effect）
临近交割月时持仓逐渐移仓，流动性下降、价差扩大，因子信号可能退化。  
以数据集最后 **30 个交易日** 为"临近交割期"，与其余时段的 IC 对比。

In [ ]:
NEAR_DELIVERY_DAYS = 30   # 临近交割定义（交易日数）

daily_closes = klines['close'].resample('1D').last().dropna()
trading_days = daily_closes.index.sort_values()
cutoff = trading_days[-NEAR_DELIVERY_DAYS]

near_mask_daily  = trading_days >= cutoff
far_mask_daily   = trading_days < cutoff

print(f"临近交割期: {cutoff.date()} ~ {trading_days[-1].date()}  ({NEAR_DELIVERY_DAYS} 交易日)")
print(f"非临近期:   {trading_days[0].date()} ~ {(cutoff - pd.Timedelta(days=1)).date()}  "
      f"({len(far_mask_daily[far_mask_daily])} 交易日)")

fwd_ret = klines['close'].pct_change(60).shift(-60)
daily_f = factor_df.resample('1D').last()
daily_r = fwd_ret.resample('1D').last()

roll_records = []
for period, mask in [('临近交割', near_mask_daily), ('非临近期', far_mask_daily)]:
    idx = trading_days[mask]
    f_sub = daily_f.loc[daily_f.index.isin(idx)]
    r_sub = daily_r.loc[daily_r.index.isin(idx)]
    for col in daily_f.columns:
        aligned = pd.concat([f_sub[col], r_sub], axis=1).dropna()
        if len(aligned) < 5:
            continue
        rho, _ = stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
        roll_records.append({'period': period, 'factor': col, 'IC': rho})

roll_df = (pd.DataFrame(roll_records)
            .pivot(index='factor', columns='period', values='IC'))
roll_df['IC变化'] = (roll_df['临近交割'] - roll_df['非临近期']).round(4)
roll_df['退化'] = roll_df['IC变化'].abs() > 0.05

print("\nIC 对比（forward 60 bar）:")
print(roll_df.sort_values('IC变化').round(4).to_string())

# 分组柱状图对比
fig = go.Figure()
for period, color in [('非临近期', '#1f77b4'), ('临近交割', '#ff7f0e')]:
    fig.add_trace(go.Bar(
        name=period,
        x=roll_df.index,
        y=roll_df[period],
        marker_color=color,
    ))
fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.update_layout(
    barmode='group',
    title=f'换月效应：临近交割 vs 非临近期 IC 对比（forward 60 bar | 最后 {NEAR_DELIVERY_DAYS} 交易日为临近期）',
    xaxis_title='因子', yaxis_title='Rank IC', height=430
)
fig.show()

## 14. 分组 T 检验（Q5 vs Q1 显著性）
分层柱状图只有视觉，没有统计检验。对每个因子做 **Welch's t-test**（不假设等方差）：  
零假设 H₀：Q5 与 Q1 的前瞻收益均值相等。p < 0.05 视为显著，p < 0.01 视为强显著。

In [ ]:
FORWARD_BARS = 60
N_QUANTILES  = 5

def q5_q1_ttest(factor_series: pd.Series, klines: pd.DataFrame,
                forward_bars: int = 60, n_quantiles: int = 5) -> dict:
    fwd_ret = klines['close'].pct_change(forward_bars).shift(-forward_bars)
    daily_last = pd.concat([factor_series, fwd_ret], axis=1).dropna()
    daily_last = daily_last.resample('1D').last().dropna()
    daily_last.columns = ['factor', 'fwd_ret']
    daily_last['q'] = pd.qcut(daily_last['factor'], n_quantiles,
                               labels=False, duplicates='drop') + 1

    q5 = daily_last.loc[daily_last['q'] == n_quantiles, 'fwd_ret']
    q1 = daily_last.loc[daily_last['q'] == 1,           'fwd_ret']
    if len(q5) < 3 or len(q1) < 3:
        return {}

    t_stat, p_val = stats.ttest_ind(q5, q1, equal_var=False)
    diff_mean = q5.mean() - q1.mean()
    # 95% CI of the mean difference via t-distribution
    se = np.sqrt(q5.std()**2 / len(q5) + q1.std()**2 / len(q1))
    dof = (q5.std()**2/len(q5) + q1.std()**2/len(q1))**2 / (
           (q5.std()**2/len(q5))**2/(len(q5)-1) +
           (q1.std()**2/len(q1))**2/(len(q1)-1))
    ci95 = stats.t.ppf(0.975, dof) * se

    return {
        'Q5_mean':   q5.mean(),
        'Q1_mean':   q1.mean(),
        'Q5-Q1':     diff_mean,
        'CI95':      ci95,
        't_stat':    t_stat,
        'p_value':   p_val,
        'n_Q5':      len(q5),
        'n_Q1':      len(q1),
    }


ttest_records = {}
for col in factor_df.columns:
    r = q5_q1_ttest(factor_df[col], klines, FORWARD_BARS, N_QUANTILES)
    if r:
        ttest_records[col] = r

ttest_df = pd.DataFrame(ttest_records).T.sort_values('p_value')

def sig_label(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return 'ns'

ttest_df['sig'] = ttest_df['p_value'].apply(sig_label)

print(f"分组 T 检验结果（forward {FORWARD_BARS} bar，{N_QUANTILES} 分位）")
print(f"{'因子':<25} {'Q5-Q1':>10} {'CI95±':>8} {'t':>8} {'p':>8} {'显著性':>6}")
print('-' * 72)
for name, row in ttest_df.iterrows():
    print(f"{name:<25} {row['Q5-Q1']:>10.4%} {row['CI95']:>8.4%} "
          f"{row['t_stat']:>8.3f} {row['p_value']:>8.4f} {row['sig']:>6}")

# 可视化：Q5-Q1 均值差 + 95% CI，按显著性着色
colors = {'***': '#1a7f37', '**': '#2ca02c', '*': '#98df8a', 'ns': '#d62728'}
bar_colors = [colors[ttest_df.loc[n, 'sig']] for n in ttest_df.index]

fig = go.Figure(go.Bar(
    x=ttest_df.index,
    y=ttest_df['Q5-Q1'] * 100,
    error_y=dict(type='data', array=ttest_df['CI95'].values * 100, visible=True),
    marker_color=bar_colors,
    text=ttest_df['sig'],
    textposition='outside',
))
fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.update_layout(
    title=f'Q5−Q1 均值差 ± 95% CI（forward {FORWARD_BARS} bar | 深绿=p<0.01 浅绿=p<0.05 红=不显著）',
    xaxis_title='因子', yaxis_title='Q5−Q1 平均收益 (%)', height=430
)
fig.show()